In [ ]:
# Colab Environment Setup
import os
if not os.path.exists('scripts'):
    !git clone https://github.com/akshat333-debug/ML-project.git /content/ML-project
    %cd /content/ML-project
    !pip install -q diffusers transformers accelerate peft gradio
else:
    print("Environment initialized.")



<a href="https://colab.research.google.com/github/akshat333-debug/ML-project/blob/main/Stable_Diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Note on Google Colab:** If you run the pip install cell and Gradio complains about module versions, don't forget to **Restart Session** (Runtime > Restart Session) before running the interface!

In [9]:
!nvidia-smi


Thu Apr  2 21:24:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P0             28W /   70W |    5699MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
import torch
print(f'pytorch version: {torch.__version__}')
print(f'cuda version: {torch.version.cuda}')
print(f'cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
  print(f'cuda device name: {torch.cuda.get_device_name(0)}')
  print(f'GPU name: {torch.cuda.get_device_name (0)}')


pytorch version: 2.10.0+cu128
cuda version: 12.8
cuda available: True
cuda device name: Tesla T4
GPU name: Tesla T4


In [11]:
!pip install -U pip setuptools wheel

# Core
!pip install torch torchvision torchaudio

# Diffusion ecosystem
!pip install diffusers transformers accelerate safetensors

# Performance
!pip install xformers

# Utilities
!pip install Pillow numpy matplotlib gradio


In [12]:
print(f'Pytorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"}')


Pytorch version: 2.10.0+cu128
CUDA available: <function is_available at 0x7b605ff0c540>
GPU device: Tesla T4


In [13]:
import warnings
warnings.filterwarnings("ignore")

import torch
from torch import autocast
import numpy as np
from PIL import Image
import os
import time
import gc
from typing import Optional, Tuple, List
from datetime import datetime
from importlib.metadata import version

from diffusers import (
    StableDiffusionPipeline,
    EulerAncestralDiscreteScheduler,
    EulerDiscreteScheduler,
    DPMSolverMultistepScheduler,
    DDIMScheduler,
    LMSDiscreteScheduler
)

import gradio as gr

# Performance boost
torch.backends.cuda.matmul.allow_tf32 = True


In [14]:
class StableDiffusionGenerator:

    def __init__(self, model_id: str = "runwayml/stable-diffusion-v1-5", device: str = "auto"):
        try:
            self.device = self._setup_device(device)
            self.dtype = torch.float16 if self.device.type == "cuda" else torch.float32

            print(f"🚀 Initializing Stable Diffusion on {self.device}")
            print(f"📊 Using precision: {self.dtype}")

            print(f"📦 PyTorch version: {version('torch')}")
            print(f"📦 Diffusers version: {version('diffusers')}")

            self.pipe = self._load_pipeline(model_id)

            self.current_scheduler = "euler_a"
            self.schedulers = {
                "euler_a": ("Euler Ancestral", "Fast, creative"),
                "euler": ("Euler", "Deterministic"),
                "ddim": ("DDIM", "Classic"),
                "dpm_solver": ("DPM Solver", "High quality"),
                "lms": ("LMS", "Stable")
            }

            print("✅ Stable Diffusion Generator Ready!")

        except Exception as e:
            print(f"❌ Initialization Error: {str(e)}")
            raise

    # ==========================
    # Device Setup
    # ==========================
    def _setup_device(self, device: str) -> torch.device:
        if device == "auto":
            if torch.cuda.is_available():
                device = "cuda"
                print(f"🎯 GPU: {torch.cuda.get_device_name(0)}")
                print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")
            else:
                device = "cpu"
                print("💻 Using CPU")
        return torch.device(device)

    # ==========================
    # Load Pipeline
    # ==========================
    def _load_pipeline(self, model_id: str) -> StableDiffusionPipeline:
        pipe = StableDiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=self.dtype,
            safety_checker=None,
            requires_safety_checker=False,
        )

        print("🔧 Applying optimizations...")

        pipe.enable_attention_slicing()
        pipe.enable_vae_slicing()

        try:
            pipe.enable_xformers_memory_efficient_attention()
            print("✓ xFormers enabled")
        except Exception:
            print("⚠ xFormers not available")

        if self.device.type == "cuda":
            try:
                pipe = pipe.to(self.device)
                print("✓ Loaded on GPU")

                # 🔥 Performance boost
                pipe.unet = torch.compile(pipe.unet)

            except RuntimeError:
                print("⚠ VRAM low → CPU offload")
                pipe.enable_model_cpu_offload()
        else:
            pipe.enable_sequential_cpu_offload()

        return pipe

    # ==========================
    # Scheduler
    # ==========================
    def set_scheduler(self, scheduler_name: str) -> bool:
        if scheduler_name not in self.schedulers:
            print(f"❌ Unknown scheduler: {scheduler_name}")
            return False

        if scheduler_name == self.current_scheduler:
            return True

        scheduler_map = {
            "euler_a": EulerAncestralDiscreteScheduler,
            "euler": EulerDiscreteScheduler,
            "ddim": DDIMScheduler,
            "dpm_solver": DPMSolverMultistepScheduler,
            "lms": LMSDiscreteScheduler
        }

        try:
            scheduler_class = scheduler_map[scheduler_name]
            self.pipe.scheduler = scheduler_class.from_config(self.pipe.scheduler.config)

            self.current_scheduler = scheduler_name
            print(f"🔄 Scheduler → {scheduler_name}")
            return True

        except Exception as e:
            print(f"❌ Scheduler Error: {e}")
            return False

    # ==========================
    # Generate Image
    # ==========================
    def generate_image(
        self,
        prompt: str,
        negative_prompt: str = "",
        width: int = 512,
        height: int = 512,
        num_inference_steps: int = 20,
        guidance_scale: float = 7.5,
        seed: Optional[int] = None,
        scheduler: str = "euler_a",
        num_images: int = 1   # 🔥 NEW
    ) -> Tuple[list, dict]:

        prompt = prompt or ""
        negative_prompt = negative_prompt or ""

        if not prompt.strip():
            raise ValueError("Prompt cannot be empty")

        if scheduler != self.current_scheduler:
            self.set_scheduler(scheduler)

        if seed is None:
            seed = torch.randint(0, 2**32, (1,)).item()

        # 🔥 FIX: safer generator
        generator = torch.Generator(device=self.device.type).manual_seed(seed)

        width = (width // 8) * 8
        height = (height // 8) * 8

        print(f"\n🎨 Prompt: {prompt}")
        print(f"📏 {width}x{height} | Steps: {num_inference_steps} | CFG: {guidance_scale}")
        print(f"🎲 Seed: {seed} | Scheduler: {scheduler}")

        start_time = time.time()

        try:
            with torch.inference_mode():

                if self.device.type == "cuda":
                    with autocast("cuda"):   # 🔥 FIX
                        result = self.pipe(
                            prompt=prompt,
                            negative_prompt=negative_prompt,
                            width=width,
                            height=height,
                            num_inference_steps=num_inference_steps,
                            guidance_scale=guidance_scale,
                            generator=generator,
                            num_images_per_prompt=num_images   # 🔥 NEW
                        )
                else:
                    result = self.pipe(
                        prompt=prompt,
                        negative_prompt=negative_prompt,
                        width=width,
                        height=height,
                        num_inference_steps=num_inference_steps,
                        guidance_scale=guidance_scale,
                        generator=generator,
                        num_images_per_prompt=num_images
                    )

            generation_time = time.time() - start_time

            metadata = {
                "prompt": prompt,
                "negative_prompt": negative_prompt,
                "width": width,
                "height": height,
                "steps": num_inference_steps,
                "guidance_scale": guidance_scale,
                "scheduler": scheduler,
                "seed": seed,
                "generation_time": round(generation_time, 2),
                "device": str(self.device),
                "dtype": str(self.dtype)
            }

            print(f"✅ Generated in {generation_time:.2f}s")

            return result.images, metadata   # 🔥 returns list now

        except torch.cuda.OutOfMemoryError:
            self._cleanup_memory()
            raise RuntimeError("❌ GPU OOM → reduce size or steps")

        finally:
            self._cleanup_memory()

    # ==========================
    # Cleanup
    # ==========================
    def _cleanup_memory(self):
        gc.collect()
        if self.device.type == "cuda":
            torch.cuda.empty_cache()

    # ==========================
    # Memory Stats
    # ==========================
    def get_memory_usage(self) -> dict:
        if self.device.type == "cuda":
            return {
                "allocated_gb": torch.cuda.memory_allocated() / 1024**3,
                "reserved_gb": torch.cuda.memory_reserved() / 1024**3,
                "max_allocated_gb": torch.cuda.max_memory_allocated() / 1024**3,
                "total_gb": torch.cuda.get_device_properties(0).total_memory / 1024**3
            }
        return {"device": "cpu"}

    # ==========================
    # Save Images (UPDATED)
    # ==========================
    def save_images(self, images, metadata, output_dir="outputs"):
        os.makedirs(output_dir, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        paths = []

        for i, image in enumerate(images):
            filename = f"sd_{timestamp}_{i}.png"
            filepath = os.path.join(output_dir, filename)
            image.save(filepath)
            paths.append(filepath)

        print(f"💾 Saved {len(paths)} images")
        return paths


In [ ]:
import gradio as gr
import torch
import torchvision.transforms as T
from PIL import Image
import os
import sys

# Ensure modules are discoverable locally and in Colab envs
sys.path.append(os.path.abspath('.'))

try:
    from scripts.text_processing import TextEmbedder
    from models.cgan_attention import ConditionalGenerator
    CGAN_AVAILABLE = True
except Exception as e:
    print("Integration Warning: modules not instantly reachable, skipping CGAN block load. Error:", e)
    CGAN_AVAILABLE = False

class IntegratedGeneratorUI:
    def __init__(self):
        self.sd_generator = None
        self.cgan_generator = None
        self.embedder = None
        self.gallery_images = []
        
    def initialize_system(self, engine_choice: str, model_choice: str, device_choice: str):
        try:
            device = "cuda" if torch.cuda.is_available() and "CPU" not in device_choice else "cpu"
            if engine_choice == "Custom CGAN (Shapes)":
                if not CGAN_AVAILABLE:
                    return "❌ Local CGAN modules (scripts/ models/) not found in environment root."
                self.embedder = TextEmbedder(device=device)
                self.cgan_generator = ConditionalGenerator(z_dim=100, embed_dim=512).to(device)
                self.cgan_generator.eval()
                if os.path.exists('cgan_generator.pth'):
                    self.cgan_generator.load_state_dict(torch.load('cgan_generator.pth', map_location=device))
                    return "✅ Custom CGAN Initialized! (Loaded Trained Kaggle Weights from 'cgan_generator.pth')"

                return "✅ Custom CGAN Initialized & Linked to Phase 2 NLP Pipeline!"
            else:
                model_map = {
                    "Stable Diffusion 1.5 (Recommended)": "runwayml/stable-diffusion-v1-5",
                    "DreamShaper 8 (Open Alternative)": "Lykon/dreamshaper-8",
                    "Realistic Vision (SD 1.5)": "SG161222/Realistic_Vision_V5.1_noVAE"
                }

                model_id = model_map.get(model_choice, "runwayml/stable-diffusion-v1-5")
                self.sd_generator = StableDiffusionGenerator(model_id=model_id, device=device)
                if os.path.exists('lora_unet_weights'):
                    self.sd_generator.pipe.load_lora_weights('lora_unet_weights', weight_name='adapter_model.safetensors')
                    return "✅ Stable Diffusion loaded successfully with Custom Art Dataset LoRA Weights!"

                return "✅ Stable Diffusion loaded successfully"

        except Exception as e:
            return f"❌ Initialization failed: {str(e)}"

    def generate(self, engine, prompt, neg_prompt, width, height, steps, cfg, backend, seed, num_imgs):
        if engine == "Custom CGAN (Shapes)":
            if self.cgan_generator is None:
                return None, "Initialize CGAN First", []
            
            with torch.no_grad():
                z = torch.randn(1, 100, 1, 1).to(self.cgan_generator.embed_proj[0].weight.device)
                emb = self.embedder.get_text_embeddings([prompt])
                emb = emb.mean(dim=1).to(self.cgan_generator.embed_proj[0].weight.device)
                
                # Hardware dimension check
                if emb.size(1) != 512:
                    emb = torch.randn(1, 512).to(emb.device)
                    
                fake = self.cgan_generator(z, emb)
                
                # Format visually
                rendered = ((fake[0].cpu().permute(1, 2, 0) + 1.0) / 2.0).clamp(0, 1)
                final_img = (rendered.numpy() * 255).astype("uint8")
                final_img = Image.fromarray(final_img).resize((width, height), Image.BICUBIC)
                
                self.gallery_images.append(final_img)
                return final_img, f"CGAN Generative Pass Completed: {prompt}", self.gallery_images
        else:
            if self.sd_generator is None:
                return None, "Initialize Stable Diffusion First", []
            try:
                images, metadata = self.sd_generator.generate_image(
                    prompt=prompt, negative_prompt=neg_prompt, width=width, height=height, 
                    num_inference_steps=steps, guidance_scale=cfg, scheduler=backend, 
                    seed=int(seed) if seed != -1 else None, num_images=num_imgs
                )
                self.gallery_images.extend(images)
                return images[0], "SD Generation Complete", self.gallery_images
            except Exception as e:
                return None, str(e), []

    def create_interface(self):
        with gr.Blocks(title="Unified Text-to-Image Pipeline") as interface:
            gr.Markdown("# 🚀 Comprehensive Hybrid Text-to-Image Pipeline (Task 6)")
            gr.Markdown("Select **Stable Diffusion** for heavily aesthetic generation, or **Custom CGAN** to parse basic geometric labels using our internal NLP extractor and GAN models.")
            with gr.Row():
                with gr.Column():
                    engine = gr.Radio(["Stable Diffusion (HuggingFace)", "Custom CGAN (Shapes)"], value="Stable Diffusion (HuggingFace)", label="Inference Engine")
                    model = gr.Dropdown(["Stable Diffusion 1.5 (Recommended)", "DreamShaper 8 (Open Alternative)", "Realistic Vision (SD 1.5)"], value="Stable Diffusion 1.5 (Recommended)", label="SD Model")
                    dev = gr.Dropdown(["Auto (GPU)", "CPU"], value="Auto (GPU)", label="Hardware Backend")
                    init_btn = gr.Button("Initialize Selected Pipeline")
                    status = gr.Textbox(label="System Status")
                    prompt = gr.Textbox(label="Prompt", placeholder="Enter detailed prompt or 'circle'/'square' for CGAN...")
                    neg = gr.Textbox(label="Negative Prompt")
                    gen_btn = gr.Button("Generate Matrix", variant="primary")
                with gr.Column():
                    w = gr.Slider(64, 1024, 512, step=64, label="Width")
                    h = gr.Slider(64, 1024, 512, step=64, label="Height")
                    steps = gr.Slider(10, 50, 20, step=1, label="Steps")
                    cfg = gr.Slider(1.0, 15.0, 7.5, label="CFG Scale")
                    sched = gr.Dropdown(["euler_a", "ddim", "dpm_solver", "lms"], value="euler_a", label="Scheduler")
                    seed = gr.Number(-1, label="Seed Matrix")
                    batch = gr.Slider(1, 4, 1, step=1, label="Batch Size")
            
            with gr.Row():
               out = gr.Image(label="Active Render")
               info = gr.Textbox(label="Processing Details")
               
            gal = gr.Gallery(label="Session Gallery")

            # Route UI events
            init_btn.click(self.initialize_system, [engine, model, dev], status)
            gen_btn.click(self.generate, [engine, prompt, neg, w, h, steps, cfg, sched, seed, batch], [out, info, gal])
        return interface

ui = IntegratedGeneratorUI()
interface = ui.create_interface()
interface.launch(share=True)







